In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [2]:
train = pd.read_csv('./data/train_reviews.csv')
test = pd.read_csv('./data/test_reviews.csv')
negocios = pd.read_csv('./data/negocios.csv')
usuarios = pd.read_csv('./data/usuarios.csv')

C:\Users\Lluis\AppData\Local\Temp\ipykernel_9668\4243529203.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios = pd.read_csv('./data/usuarios.csv')


In [3]:
def generateSubmision(test_preds, name):
    submition = pd.DataFrame()
    submition['review_id'] = test['review_id']
    submition['stars'] = test_preds
    submition.to_csv(f'./data/submissions/submission_{name}.csv', index=False)

In [4]:
import torch
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [22]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


In [23]:
import evaluate

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [24]:
from peft import get_peft_model, LoraConfig, TaskType    
peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,  # Rank of the update matrices
        lora_alpha=32,  # Alpha parameter for scaling
        lora_dropout=0.1,  # Dropout probability for LoRA layers
        target_modules=["query", "value"]  # Target specific modules (for DistilBERT)
    )

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 298,757 || all params: 167,659,018 || trainable%: 0.1782


In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long) - 1  # Convert 1-5 to 0-4
        }

In [26]:
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(train, test_size=0.2, random_state=42)

In [27]:
train_dataset = SentimentDataset(
    train_df['text'].tolist(),
    train_df['stars'].tolist(),
    tokenizer
)

eval_dataset = SentimentDataset(
    eval_df['text'].tolist(),
    eval_df['stars'].tolist(),
    tokenizer
)


In [28]:
from transformers import Trainer, TrainingArguments
output_dir = "./model_output"
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [29]:
if torch.cuda.is_available():
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Training on CPU")
torch.cuda.empty_cache()


Training on GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.669400,0.662566,0.722278


TrainOutput(global_step=48390, training_loss=0.6888946835919337, metrics={'train_runtime': 3845.7103, 'train_samples_per_second': 201.322, 'train_steps_per_second': 12.583, 'total_flos': 5.110593532649318e+16, 'train_loss': 0.6888946835919337, 'epoch': 1.0})

In [32]:
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('./model_output\\tokenizer_config.json',
 './model_output\\special_tokens_map.json',
 './model_output\\vocab.txt',
 './model_output\\added_tokens.json',
 './model_output\\tokenizer.json')

In [51]:
import torch
from torch.nn.functional import softmax
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Inference settings
batch_size =4
texts = test["text"].tolist()
all_labels = []
all_scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]

    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # No gradient computation for inference
    with torch.no_grad():
        outputs = model(**inputs)
        # probs = softmax(outputs.logits, dim=1)
        # scores, labels = torch.max(probs, dim=1)
        # print(outputs)
        # predictions = torch.argmax(outputs.logits, dim=1).item() + 1
        probs = softmax(outputs.logits, dim=1)
        scores, labels = torch.max(probs, dim=1)
        labels = labels + 1
    all_labels.extend(labels.cpu().numpy().tolist())
    all_scores.extend(scores.cpu().numpy().tolist())


100%|██████████| 103692/103692 [53:38<00:00, 32.22it/s] 


In [53]:
import pickle
with open('./data/submissions/labels.pkl', 'wb') as f:
    pickle.dump(all_labels, f)
with open('./data/submissions/scores.pkl', 'wb') as f:
    pickle.dump(all_scores, f)

In [2]:
import pickle
with open('./data/submissions/scores.pkl', 'rb') as f:
    all_scores = pickle.load(f)

In [52]:
generateSubmision(all_labels, 'lora_bert_base_multilingual_uncased_sentiment')